## Ingest Customer Dimension Data into Bronze Layer

   Import required libraries

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType 

In [0]:
%run /Workspace/Shopvista/Shopvista-Data-Engineering-Project-_databricks/1_setup/utilities

In [0]:
print (bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "shopvista", "catalog")
dbutils.widgets.text("data_source", "customers", "data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source=dbutils.widgets.get("data_source")

print("category", "data_source")

In [0]:

# Define schema for the customer data

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True)
])


In [0]:

#  show the path to the datasource 

base_path = f"s3://shopvista-sv/{data_source}/*.csv"
print(base_path)

In [0]:
#load data into dataframe and add metadata

df = (
    spark.read.format("csv")
        .option("header",True)
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp()) # add metadata column
        .select("*", "_metadata.file_name", "_metadata.file_path") #add metadata column

)
display(df.limit(10))

In [0]:
df.printSchema()

   ### write customer data to bronze schema 

In [0]:
# write customer to bronze table

df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")    